In [ ]:
import os
import pandas as pd
from IPython.display import FileLink, display

backup_dir = "/kaggle/working/HAR_Research"
os.makedirs(backup_dir, exist_ok=True)

manifest_path = os.path.join(
    backup_dir,
    "ucf101_split01_manifest.csv"
)

df.to_csv(manifest_path, index=False)

print("Manifest backup created.")
print("Total videos:", len(df))
print(df["split"].value_counts())

display(FileLink(manifest_path))

In [ ]:

import time
import torch
from tqdm.auto import tqdm

def train_one_epoch(loader):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    start = time.time()

    progress = tqdm(loader, desc="Training Epoch 1")

    for videos, labels in progress:

        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Uniform selection: 4 frames from 29 candidates
        videos = uniform_sampling(videos, budget=4)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += batch_size

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            accuracy=f"{100 * correct / total:.2f}%"
        )

    return {
        "train_loss": total_loss / total,
        "train_accuracy": 100 * correct / total,
        "training_seconds": time.time() - start
    }


# Train epoch 1
results = train_one_epoch(train_loader)

print("\nTRAINING COMPLETED")
print("Training loss:", round(results["train_loss"], 4))
print("Training accuracy:", round(results["train_accuracy"], 2), "%")
print("Training time:", round(results["training_seconds"], 2), "seconds")

# Save checkpoint immediately
save_checkpoint(
    model,
    optimizer,
    epoch=1,
    filename="mobilenetv3_uniform_4frames_epoch1.pth"
)

print("\nFIRST BASELINE CHECKPOINT SAVED.")



In [ ]:

import os
import zipfile
import hashlib

checkpoint_path = (
    "/kaggle/working/HAR_Research/checkpoints/"
    "mobilenetv3_uniform_4frames_epoch1.pth"
)

assert os.path.exists(checkpoint_path), "Checkpoint missing!"

size_mb = os.path.getsize(checkpoint_path) / (1024 * 1024)

# Calculate SHA256 checksum for backup verification
sha256 = hashlib.sha256()

with open(checkpoint_path, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha256.update(chunk)

print("CHECKPOINT VERIFIED")
print("File:", os.path.basename(checkpoint_path))
print("Size:", round(size_mb, 2), "MB")
print("SHA256:", sha256.hexdigest())

# Create a ZIP backup
zip_path = (
    "/kaggle/working/"
    "mobilenetv3_uniform_4frames_epoch1_backup.zip"
)

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
    z.write(
        checkpoint_path,
        arcname=os.path.basename(checkpoint_path)
    )

print("\nZIP BACKUP CREATED")
print("Path:", zip_path)
print("Size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")


In [37]:

import torch
from tqdm.auto import tqdm

# Evaluate validation set
model.eval()

total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():

    for videos, labels in tqdm(val_loader, desc="Validation"):

        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # Same 4-frame uniform sampling protocol
        videos = uniform_sampling(videos, budget=4)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)

        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

val_loss = total_loss / total
val_accuracy = 100 * correct / total

print("\nVALIDATION RESULTS")
print("------------------")
print("Validation videos:", total)
print("Validation loss:", round(val_loss, 4))
print("Validation accuracy:", round(val_accuracy, 2), "%")


Validation:   0%|          | 0/802 [00:00<?, ?it/s]


VALIDATION RESULTS
------------------
Validation videos: 1604
Validation loss: 1.8577
Validation accuracy: 53.87 %


In [39]:

# Train the second epoch
results_epoch2 = train_one_epoch(train_loader)

print("\nEPOCH 2 TRAINING RESULTS")
print("Training loss:", round(results_epoch2["train_loss"], 4))
print("Training accuracy:", round(results_epoch2["train_accuracy"], 2), "%")
print("Training time:", round(results_epoch2["training_seconds"], 2), "seconds")

# Save checkpoint immediately
save_checkpoint(
    model,
    optimizer,
    epoch=2,
    filename="mobilenetv3_uniform_4frames_epoch2.pth"
)

print("\nEPOCH 2 CHECKPOINT SAVED")



Training Epoch 1:   0%|          | 0/3967 [00:00<?, ?it/s]


EPOCH 2 TRAINING RESULTS
Training loss: 0.9614
Training accuracy: 73.68 %
Training time: 948.29 seconds
Checkpoint saved: /kaggle/working/HAR_Research/checkpoints/mobilenetv3_uniform_4frames_epoch2.pth
Size (MB): 49.92

EPOCH 2 CHECKPOINT SAVED


In [41]:
model.eval()

total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for videos, labels in tqdm(val_loader, desc="Validation Epoch 2"):
        videos = videos.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        videos = uniform_sampling(videos, budget=4)

        outputs = model(videos)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)

val_loss = total_loss / total
val_accuracy = 100 * correct / total

print("\nEPOCH 2 VALIDATION RESULTS")
print(f"Validation loss: {val_loss:.4f}")
print(f"Validation accuracy: {val_accuracy:.2f}%")
print(f"Validation samples: {total}")


Validation Epoch 2:   0%|          | 0/802 [00:00<?, ?it/s]


EPOCH 2 VALIDATION RESULTS
Validation loss: 1.8382
Validation accuracy: 63.47%
Validation samples: 1604


In [42]:
import os
import zipfile
import hashlib

results_path = "/kaggle/working/HAR_Research/results/baseline_experiment_results.csv"

assert os.path.exists(results_path), "Results CSV not found!"

zip_path = "/kaggle/working/baseline_experiment_results_backup.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write(results_path, arcname="baseline_experiment_results.csv")

sha256 = hashlib.sha256(open(results_path, "rb").read()).hexdigest()

print("Backup created:", zip_path)
print("CSV size:", round(os.path.getsize(results_path) / 1024, 2), "KB")
print("SHA256:", sha256)

                 experiment  epoch  candidate_frames  selected_frames  train_samples  validation_samples  train_loss  train_accuracy  validation_loss  validation_accuracy  training_time_seconds
MobileNetV3_Uniform_4Frames      1                29                4           7933                1604      2.8082           33.48         1.857700            53.870000                 936.15
MobileNetV3_Uniform_4Frames      2                29                4           7933                1604      0.9614           73.68         1.838208            63.466334                 948.29

Results saved: /kaggle/working/HAR_Research/results/baseline_experiment_results.csv


In [36]:
# Get one batch
images, labels = next(iter(experiment2_train_loader))

# Move data to GPU
images = images.to(device)
labels = labels.to(device)

# Forward pass
exp2_model.train()

with torch.no_grad():
    outputs = exp2_model(images)

print("Experiment 2 batch test successful!")
print("Input shape:", images.shape)
print("Labels shape:", labels.shape)
print("Output shape:", outputs.shape)
print("Device:", images.device)
print("Loss:", exp2_criterion(outputs, labels).item())

Experiment 2 batch test successful!
Input shape: torch.Size([2, 29, 3, 224, 224])
Labels shape: torch.Size([2])
Output shape: torch.Size([2, 101])
Device: cuda:0
Loss: 4.577654838562012


In [47]:

import os
import time
import torch
import pandas as pd
from tqdm.auto import tqdm

# Training configuration
EXP2_EPOCHS = 8
EXP2_PATIENCE = 3
EXP2_HISTORY = []
EXP2_BEST_VAL_ACC = 0.0
epochs_without_improvement = 0

EXP2_BEST_CHECKPOINT = os.path.join(
    EXP2_CHECKPOINT_DIR,
    "experiment2_best_model.pth"
)

EXP2_LATEST_CHECKPOINT = os.path.join(
    EXP2_CHECKPOINT_DIR,
    "experiment2_latest_checkpoint.pth"
)

# Training loop
for epoch in range(EXP2_EPOCHS):

    start_time = time.time()

    # =========================
    # 1. TRAINING
    # =========================
    exp2_model.train()

    train_loss_total = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(
        experiment2_train_loader,
        desc=f"Experiment 2 - Epoch {epoch + 1}/{EXP2_EPOCHS}"
    )

    for images, labels in train_bar:

        images = images.to(device)
        labels = labels.to(device)

        exp2_optimizer.zero_grad(set_to_none=True)

        outputs = exp2_model(images)
        loss = exp2_criterion(outputs, labels)

        loss.backward()
        exp2_optimizer.step()

        train_loss_total += loss.item() * labels.size(0)

        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

        train_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    train_loss = train_loss_total / train_total
    train_accuracy = 100.0 * train_correct / train_total

    # =========================
    # 2. VALIDATION
    # =========================
    exp2_model.eval()

    val_loss_total = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in tqdm(
            experiment2_val_loader,
            desc="Validation",
            leave=False
        ):

            images = images.to(device)
            labels = labels.to(device)

            outputs = exp2_model(images)
            loss = exp2_criterion(outputs, labels)

            val_loss_total += loss.item() * labels.size(0)

            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_total / val_total
    val_accuracy = 100.0 * val_correct / val_total

    epoch_time = time.time() - start_time

    # =========================
    # 3. SAVE RESULTS
    # =========================
    result = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "learning_rate": exp2_optimizer.param_groups[0]["lr"],
        "training_time_seconds": epoch_time
    }

    EXP2_HISTORY.append(result)

    print("\n" + "=" * 45)
    print(f"EXPERIMENT 2 - EPOCH {epoch + 1}")
    print("=" * 45)
    print(f"Training Loss:       {train_loss:.4f}")
    print(f"Training Accuracy:   {train_accuracy:.2f}%")
    print(f"Validation Loss:     {val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.2f}%")
    print(f"Training Time:       {epoch_time:.2f} seconds")

    # =========================
    # 4. UPDATE LR SCHEDULER
    # =========================
    exp2_scheduler.step(val_accuracy)

    # =========================
    # 5. SAVE BEST MODEL
    # =========================
    if val_accuracy > EXP2_BEST_VAL_ACC:

        EXP2_BEST_VAL_ACC = val_accuracy
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": exp2_model.state_dict(),
            "optimizer_state_dict": exp2_optimizer.state_dict(),
            "best_val_accuracy": EXP2_BEST_VAL_ACC,
            "history": EXP2_HISTORY
        }, EXP2_BEST_CHECKPOINT)

        print("Best model checkpoint saved.")

    else:
        epochs_without_improvement += 1

    # =========================
    # 6. SAVE LATEST CHECKPOINT
    # =========================
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": exp2_model.state_dict(),
        "optimizer_state_dict": exp2_optimizer.state_dict(),
        "scheduler_state_dict": exp2_scheduler.state_dict(),
        "best_val_accuracy": EXP2_BEST_VAL_ACC,
        "history": EXP2_HISTORY
    }, EXP2_LATEST_CHECKPOINT)

    # Save results after every epoch
    pd.DataFrame(EXP2_HISTORY).to_csv(
        "/kaggle/working/experiment2_results.csv",
        index=False
    )

    print(f"Best validation accuracy: {EXP2_BEST_VAL_ACC:.2f}%")
    print(f"Checkpoint saved for epoch {epoch + 1}.")

    # =========================
    # 7. EARLY STOPPING
    # =========================
    if epochs_without_improvement >= EXP2_PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\n" + "=" * 45)
print("EXPERIMENT 2 TRAINING COMPLETED")
print("=" * 45)

print("Best validation accuracy:", f"{EXP2_BEST_VAL_ACC:.2f}%")
print("Completed epochs:", len(EXP2_HISTORY))
print("Best checkpoint:", EXP2_BEST_CHECKPOINT)
print("Latest checkpoint:", EXP2_LATEST_CHECKPOINT)

display(pd.DataFrame(EXP2_HISTORY))


Experiment 2 - Epoch 1/8:   0%|          | 0/3967 [00:00<?, ?it/s]

Validation:   0%|          | 0/802 [00:00<?, ?it/s]


EXPERIMENT 2 - EPOCH 1
Training Loss:       2.6269
Training Accuracy:   37.01%
Validation Loss:     1.8768
Validation Accuracy: 57.42%
Training Time:       2254.68 seconds
Best model checkpoint saved.
Best validation accuracy: 57.42%
Checkpoint saved for epoch 1.


Experiment 2 - Epoch 2/8:   0%|          | 0/3967 [00:00<?, ?it/s]

Validation:   0%|          | 0/802 [00:00<?, ?it/s]


EXPERIMENT 2 - EPOCH 2
Training Loss:       0.8331
Training Accuracy:   76.63%
Validation Loss:     2.0828
Validation Accuracy: 60.54%
Training Time:       2258.76 seconds
Best model checkpoint saved.
Best validation accuracy: 60.54%
Checkpoint saved for epoch 2.


... (9 outputs left out)

Experiment 2 - Epoch 6/8:   0%|          | 0/3967 [00:00<?, ?it/s]

Validation:   0%|          | 0/802 [00:00<?, ?it/s]


EXPERIMENT 2 - EPOCH 6
Training Loss:       0.0697
Training Accuracy:   98.13%
Validation Loss:     4.2615
Validation Accuracy: 63.59%
Training Time:       2273.82 seconds
Best validation accuracy: 65.71%
Checkpoint saved for epoch 6.

Early stopping triggered.

EXPERIMENT 2 TRAINING COMPLETED
Best validation accuracy: 65.71%
Completed epochs: 6
Best checkpoint: /kaggle/working/HAR_Research/experiment2_checkpoints/experiment2_best_model.pth
Latest checkpoint: /kaggle/working/HAR_Research/experiment2_checkpoints/experiment2_latest_checkpoint.pth


,epoch,train_loss,train_accuracy,val_loss,val_accuracy,learning_rate,training_time_seconds
0,1,2.626927,37.009958,1.876824,57.418953,0.00010,2254.684831
1,2,0.833084,76.629270,2.082795,60.536160,0.00010,2258.755435
2,3,0.421335,88.163368,2.121901,65.710723,0.00010,2259.624494
3,4,0.239854,93.319047,3.983345,63.092269,0.00010,2258.744259
4,5,0.169965,95.235094,4.394845,62.281796,0.00010,2260.084377
5,6,0.069721,98.134375,4.261525,63.591022,0.00005,2273.823988


## Dataset Partitioning and Leakage Prevention

The UCF101 dataset was partitioned using the official Split 1 training and testing lists.

The official training data was further divided into training and validation subsets using a group-based stratified procedure within each action class.

- **Training:** 7,933 videos
- **Validation:** 1,604 videos
- **Testing:** 3,783 videos
- **Total:** 13,320 videos
- **Action classes:** 101

A fixed random seed (`random_state=42`) was used to ensure reproducibility. Source-group overlap between the training and validation subsets was checked and confirmed to be absent.

The official test set remains separate and will be reserved for final evaluation.

**Leakage Check:** PASSED.

### Stage 2: Dataset Integrity Audit — Completed

The complete UCF101 video integrity audit was successfully repeated.

- Total videos audited: 13,320
- Fully readable videos: 13,096
- Partially readable videos: 224
- Unreadable videos: 0
- Audit duration: 772.52 seconds

All videos contain at least one successfully decoded frame. Partially readable videos are retained using their actual decoded frame counts.

**Output:** `full_video_integrity_audit.csv`

**Next:** Restore the audited manifest, video Dataset class, and DataLoaders.

### Project Recovery — November 22, 2026

The Kaggle session restarted and the previous training checkpoint was not preserved.

**Previously completed work:**
- UCF101 dataset preparation and integrity audit.
- 32-frame candidate extraction pipeline.
- Uniform and temporal-difference sampling methods.
- MobileNetV3-Large video classifier.
- First baseline training epoch.

**Previously recorded baseline results:**
- Training videos: 7,933
- Frame budget: 4
- Training loss: 2.7532
- Training accuracy: 34.35%
- Training time: 901.24 seconds

**Recovery plan:** Restore the dataset pipeline, retrain the baseline, and immediately download the model checkpoint to local storage.

### Stage 1: Dataset Preparation and Model Pipeline Validation

**Completed tasks:**
- Verified the UCF101 dataset containing 13,320 videos across 101 action classes.
- Prepared training (7,933), validation (1,604), and testing (3,783) partitions.
- Completed video integrity auditing and generated an audited dataset manifest.
- Developed and tested sequential video decoding and 32-frame candidate extraction.
- Implemented and validated two frame-selection strategies:
  - Uniform temporal sampling
  - Temporal-difference-based adaptive sampling
- Tested frame budgets of 4, 8, 12, and 16.
- Configured MobileNetV3-Large with a 101-class output layer.
- Successfully completed a 10-batch GPU training pilot.

**Initial status:** The data pipeline, sampling methods, classification model, and initial GPU training workflow are operational.

**Next stage:** Conduct controlled baseline and proposed-method experiments, followed by validation and computational-efficiency evaluation.

*Note: The pilot is a pipeline test, not a final model-performance evaluation.*